# 🎯 Prompt Exploration: Therapy Companion

**Goal**: Test system/user prompts, eval response quality for your use case (parents/romance/emotions).

Loads your config model automatically. No setup needed.

In [ ]:
# Core imports + your manager
import logging
from pathlib import Path

logging.basicConfig(level=logging.INFO)

from therapy_ai.core.model_manager import ModelManager
from therapy_ai.core.backend import BackendFactory

# Load your config
manager = ModelManager.from_config("../config/default_config.yaml")
loaded = manager.load()
model, tokenizer = loaded.model, loaded.tokenizer

print(f"✅ Loaded: {loaded.config.model_id} on {BackendFactory.detect()}")
print(f"Memory: {loaded.backend} | Max tokens: {loaded.config.max_new_tokens}")

FileNotFoundError: [Errno 2] No such file or directory: '.config/default_config.yaml'

In [ ]:
# Interactive prompt tester
from mlx_lm import generate

def test_prompt(system_prompt: str, user_prompt: str, temp=0.7, max_tokens=400):
    """Test a full conversation turn."""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]
    
    prompt = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
    response = generate(
        model, tokenizer, prompt=prompt,
        max_tokens=max_tokens,
        temp=temp,
        verbose=False
    )
    
    print("=== PROMPT ===")
    print(system_prompt[:200] + "...")
    print("\n=== USER ===")
    print(user_prompt)
    print("\n=== RESPONSE ===")
    print(response)
    print("\n" + "="*80 + "\n")
    
    return response

## Therapy Prompt Tests

Your use case: parents, emotions, romance.

In [ ]:
# Test 1: Parent relationship unpacking
system = """
You are a private reflective companion. Goals:
- Empathetic listening, non-judgmental
- Help user articulate feelings
- Ask **one** gentle reflective question
- Avoid advice unless asked
- Focus: family dynamics, emotional patterns
"""

user = "I feel resentment toward my parents but also guilt. It's confusing."
test_prompt(system, user)

In [ ]:
# Test 2: Romantic patterns
user = "I keep choosing partners who remind me of my father. Why?"
test_prompt(system, user, temp=0.8)

In [ ]:
# Test 3: Multi-turn simulation
messages = [
    {"role": "system", "content": system},
    {"role": "user", "content": "My parents criticized me a lot growing up."},
    {"role": "assistant", "content": "That sounds heavy. What kind of criticism stands out most?"},
    {"role": "user", "content": "They said I was never good enough."}
]

prompt = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
response = generate(model, tokenizer, prompt=prompt, max_tokens=300, temp=0.7)
print("=== MULTI-TURN ===")
print(response)

## Custom Prompt Playground

Tweak and re-run:

In [ ]:
# YOUR TURN: Edit these, re-run cell
MY_SYSTEM = """[Your custom system prompt here]"""
MY_USER = "[Your personal reflection here]"

test_prompt(MY_SYSTEM, MY_USER)

In [ ]:
# Cleanup
manager.unload()
print("Model unloaded.")